## 탈라세미아 검사 결과를 예측하라 (다중분류)
- 성별, 나이, 혈압, 콜레스테롤, 공복혈당, 최대 심박수, 심장마비 확률(output) 등의 컬럼이 있음
- 평가: 정확도(Accuracy), F1(macro) 를 구하시오
- target : thall (1:고정 결손, 2:정상, 3:가역 결손)

[제출 형식]

① CSV 파일명: `result.csv`

② 예측 클래스 컬럼명: pred

③ 제출 컬럼 개수: pred 컬럼 1개

④ 평가용 데이터 개수와 예측 결과 데이터 개수 일치: 60개

- 제출 CSV 파일 예시
~~~
pred
2
3
1
~~~


### 평가(다중분류)
~~~
from sklearn.metrics import accuracy_score, f1_score

# 정확도
print(accuracy_score(y_test, pred))

# F1 (다중분류: 클래스별 F1의 단순 평균)
print(f1_score(y_test, pred, average='macro'))
~~~


In [30]:
# 1. 문제 정의
# 다중 분류
# y: thall
# 제출 파일: result.csv

# 2. 데이터 불러오기
import pandas as pd
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/ex3/train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/ex3/test.csv")


In [31]:
# 3. 탐색(EDA)
print(train.shape, test.shape)

train.head()

(241, 14) (60, 13)


,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,output,thall
0,39,0,2,94,199,0,1,179,0,0.0,2,0,1,2
1,74,0,1,120,269,0,0,121,1,0.2,2,1,1,2
2,52,1,1,128,205,1,1,184,0,0.0,2,0,1,2
3,71,0,0,112,149,0,1,125,0,1.6,1,0,1,2
4,50,1,2,129,196,0,1,163,0,0.0,2,0,1,2


In [32]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241 entries, 0 to 240
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       241 non-null    int64  
 1   sex       241 non-null    int64  
 2   cp        241 non-null    int64  
 3   trtbps    241 non-null    int64  
 4   chol      241 non-null    int64  
 5   fbs       241 non-null    int64  
 6   restecg   241 non-null    int64  
 7   thalachh  241 non-null    int64  
 8   exng      241 non-null    int64  
 9   oldpeak   241 non-null    float64
 10  slp       241 non-null    int64  
 11  caa       241 non-null    int64  
 12  output    241 non-null    int64  
 13  thall     241 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 26.5 KB


In [33]:
train.describe()

,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,output,thall
count,241.000000,241.000000,241.000000,241.000000,241.000000,241.000000,241.000000,241.000000,241.000000,241.000000,241.000000,241.000000,241.000000,241.000000
mean,54.522822,0.676349,0.908714,131.082988,247.062241,0.120332,0.539419,149.734440,0.348548,1.024066,1.377593,0.788382,0.518672,2.356846
std,9.039756,0.468842,1.020604,17.752739,52.306870,0.326026,0.531803,21.693413,0.477502,1.182759,0.621286,1.053185,0.500691,0.589177
min,34.000000,0.000000,0.000000,94.000000,126.000000,0.000000,0.000000,88.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,47.000000,0.000000,0.000000,120.000000,212.000000,0.000000,0.000000,134.000000,0.000000,0.000000,1.000000,0.000000,0.000000,2.000000
50%,55.000000,1.000000,1.000000,130.000000,243.000000,0.000000,1.000000,152.000000,0.000000,0.600000,1.000000,0.000000,1.000000,2.000000
75%,61.000000,1.000000,2.000000,140.000000,274.000000,0.000000,1.000000,165.000000,1.000000,1.600000,2.000000,1.000000,1.000000,3.000000
max,77.000000,1.000000,3.000000,200.000000,564.000000,1.000000,2.000000,195.000000,1.000000,6.200000,2.000000,4.000000,1.000000,3.000000


In [34]:
train['thall'].value_counts()

,count
thall,
2,127
3,100
1,14


In [35]:
# 4. 전처리
# X/y split

y_train = train.pop('thall')
X_train = train

In [36]:
# one-hot encoding
X_train = pd.get_dummies(X_train, dtype=int)
test = pd.get_dummies(test, dtype=int)

X_train, test = X_train.align(test, join='left', axis=1, fill_value=0)

# y값은 이미 1, 2, 3이기 때문에 라벨 인코딩 생략.

In [37]:
# 5. 검증 데이터 분리
from sklearn.model_selection import train_test_split

# 영상 참고: stratify 추가. target 데이터 불균형의 경우.
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

print(X_tr.shape, y_tr.shape, X_val.shape, y_val.shape)

(192, 13) (192,) (49, 13) (49,)


In [47]:
# 6. 학습 및 평가
# 6-1. Random Forest
from sklearn.ensemble import RandomForestClassifier

# 영상 참고. class_weight='balanced' 추가. 적은 클래스에 가중치를 더 줌. 불균형 데이터일 때 사용.
rf = RandomForestClassifier(random_state=42, class_weight='balanced')
rf.fit(X_tr, y_tr)
y_pred_rf = rf.predict(X_val)

# 평가
# 위에서 주어진 코드 복사 및 수정
from sklearn.metrics import accuracy_score, f1_score

# 정확도
print(accuracy_score(y_val, y_pred_rf))

# F1 (다중분류: 클래스별 F1의 단순 평균)
print(f1_score(y_val, y_pred_rf, average='macro'))

0.6938775510204082
0.4678362573099415


In [48]:
# 6-2. LightGBM
from lightgbm import LGBMClassifier

# 영상 참고. class_weight='balanced' 추가. 적은 클래스에 가중치를 더 줌. 불균형 데이터일 때 사용.
lgbm = LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced')
lgbm.fit(X_tr, y_tr)
y_pred_lgbm = lgbm.predict(X_val)

# 평가
# 위에서 주어진 코드 복사 및 수정
from sklearn.metrics import accuracy_score, f1_score

# 정확도
print(accuracy_score(y_val, y_pred_lgbm))

# F1
print(f1_score(y_val, y_pred_lgbm, average='macro'))

0.7142857142857143
0.4869015356820235


In [50]:
# XGBoost는 타겟이 0부터 시작해야 됨. 고로 타켓이 1부터 시작하는 경우는 가급적 실수를 줄이기 위해 사용하지 말자.

In [49]:
# 7. 결과 파일
# accuracy와 f1 score 모두 더 높은 LightGBM을 최종 모델로 선정

pred = lgbm.predict(test)

# 답지 참고
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)
print(result.shape)

# 다시 읽어오기
result = pd.read_csv('result.csv')
result.head()

(60, 1)


,pred
0,2
1,2
2,3
3,1
4,2


In [41]:
# 답지 참고
# 전체 데이터로 학습
# 데이터 수가 적고 불균형 데이터인 경우 전체 데이터로 다시 학습한 뒤 test 데이터에 대해 예측 진행하기.
lgbm.fit(X_train, y_train)
pred = lgbm.predict(test)

result = pd.DataFrame({'pred': pred})
result.to_csv("result.csv", index=False)

result.head()

,pred
0,2
1,2
2,3
3,3
4,2


### 풀이

In [42]:
# 문제정의
# 평가: 정확도(Accuracy), F1(macro)
# target: thall (1:고정 결손, 2:정상, 3:가역 결손) → 다중분류, 클래스 불균형
# 최종파일: result.csv(컬럼 1개 pred, 예측 클래스)

# 라이브러리 및 데이터 불러오기
import pandas as pd
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/ex3/train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/ex3/test.csv")

# 탐색적 데이터 분석(EDA)
print("===== 데이터 크기 =====")
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

print("\n ===== 데이터 샘플 =====")
print(train.head(2))
print(test.head(2))

print("\n ===== 데이터 정보(자료형) =====")
print(train.info())

print("\n ===== train 결측치 수 =====")
print(train.isnull().sum())
print("\n ===== test 결측치 수 =====")
print(test.isnull().sum())

print("\n ===== 기초 통계 =====")
print(train.describe())

print("\n ===== 고유값 수 =====")
print(train.nunique())

print("\n ===== 타겟 분포 =====")
print(train['thall'].value_counts())


===== 데이터 크기 =====
Train Shape: (241, 14)
Test Shape: (60, 13)

 ===== 데이터 샘플 =====
   age  sex  cp  trtbps  chol  fbs  restecg  thalachh  exng  oldpeak  slp  \
0   39    0   2      94   199    0        1       179     0      0.0    2   
1   74    0   1     120   269    0        0       121     1      0.2    2   

   caa  output  thall  
0    0       1      2  
1    1       1      2  
   age  sex  cp  trtbps  chol  fbs  restecg  thalachh  exng  oldpeak  slp  \
0   48    1   1     130   245    0        0       180     0      0.2    1   
1   65    0   2     140   417    1        0       157     0      0.8    2   

   caa  output  
0    0       1  
1    1       1  

 ===== 데이터 정보(자료형) =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241 entries, 0 to 240
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       241 non-null    int64  
 1   sex       241 non-null    int64  
 2   cp        241 non-null    int64  
 3 

In [43]:
import pandas as pd
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/ex3/train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/ex3/test.csv")

# 데이터 전처리
target = train.pop('thall')
print(train.shape, test.shape)

# 검증데이터 분리
# stratify=target : 클래스 비율을 train/검증에 똑같이 유지 (불균형 데이터)
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=2022, stratify=target)
print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)
print(y_val.value_counts())

# 랜덤포레스트
# class_weight='balanced' : 적은 클래스에 가중치를 더 줌 (불균형 데이터)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

rf = RandomForestClassifier(random_state=2022, class_weight='balanced')
rf.fit(X_tr, y_tr)
pred = rf.predict(X_val)
print("랜덤포레스트")

print(accuracy_score(y_val, pred))
print(f1_score(y_val, pred, average='macro'))
print(f1_score(y_val, pred, average='micro'))

(241, 13) (60, 13)
(192, 13) (49, 13) (192,) (49,)
thall
2    26
3    20
1     3
Name: count, dtype: int64
랜덤포레스트
0.673469387755102
0.45229127432517263
0.673469387755102


In [44]:
# LightGBM
# class_weight='balanced' : 적은 클래스에 가중치를 더 줌
import lightgbm as lgb
lg = lgb.LGBMClassifier(random_state=2022, verbose=-1, class_weight='balanced')
lg.fit(X_tr, y_tr)
pred = lg.predict(X_val)
print("lightgbm")

print(accuracy_score(y_val, pred))
print(f1_score(y_val, pred, average='macro'))
print(f1_score(y_val, pred, average='micro'))


lightgbm
0.7142857142857143
0.4869015356820235
0.7142857142857143


In [45]:
# 최종제출
# F1(macro) 기준 가장 좋은 LightGBM으로 제출
pred = lg.predict(test)

result = pd.DataFrame({'pred': pred})
result.to_csv("result.csv", index=False)

# 1. pred 행의 수와 test의 행의 수 비교
print("pred: ", pred.shape)  # test 행의 수: 60

# 2. 생성한 csv 를 다시 읽어 확인
result = pd.read_csv("result.csv")
print(result['pred'].value_counts())


pred:  (60,)
pred
2    35
3    22
1     3
Name: count, dtype: int64


In [46]:
# 전체 데이터로 학습
lg.fit(train, target)
pred = lg.predict(test)

result = pd.DataFrame({'pred': pred})
result.to_csv("result.csv", index=False)

result.head()

,pred
0,2
1,2
2,3
3,1
4,2
